<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:12pt;font-weight:800;font-family:Georgia,serif;">
Gold Layer · Usine A · Génération & Enrichissement
</span><br>
<span style="font-size:9.5pt;font-family:sans-serif;color:#e8e8e8;">
Lecture depuis Silver. Génération du dataset Usine A.
Problème dominant : usure outil progressive.
KPIs, TRS, UCL/LCL, export Parquet.
</span>
</div>

<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:10pt;font-weight:700;font-family:Georgia,serif;">
00 · Imports & Configuration
</span>
</div>

In [26]:
# ============================================================
# GOLD LAYER — USINE A
# Input  : data/silver/ai4i_clean.parquet
# Output : data/gold/usine_A.parquet
#          data/gold/limites_usine_A.json
# ============================================================

import pandas as pd
import numpy as np
import json
import logging
from pathlib import Path
from datetime import datetime
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%H:%M:%S'
)
log = logging.getLogger('gold_A')

ROOT         = Path('..').resolve()
SILVER_FILE  = ROOT / 'data' / 'silver' / 'ai4i_clean.parquet'
GOLD_PATH    = ROOT / 'data' / 'gold'
GOLD_PATH.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

np.random.seed(42)

LAYOUT = dict(
    paper_bgcolor='#ffffff',
    plot_bgcolor='#ffffff',
    font=dict(family='Inter, sans-serif', color='#374151', size=11),
    hoverlabel=dict(bgcolor='#1f2937', font_color='#f9fafb', font_size=11),
)

# ── Paramètres physiques Usine A ─────────────────────────────
# N = (P × 60) / (2π × T)
PUISSANCE_W      = 6_283.0
COUPLE_NOMINAL   = 40.0
VITESSE_NOMINALE = round((PUISSANCE_W * 60) / (2 * np.pi * COUPLE_NOMINAL))

# t_cycle = 60 000 / N
TEMPS_CYCLE_NOM  = round(60_000 / VITESSE_NOMINALE, 2)

# Norme ISO 8688 — carbure acier
DUREE_VIE_OUTIL  = 250

TEMP_AIR_BASE    = 22.0
OFFSET_THERMIQUE = 10.0
REBUT_BASE       = 0.02

USINE_ID         = 'Usine_A'
MACHINES         = ['M01', 'M02', 'M03', 'M04']
TYPES_PIECES     = ['REF01', 'REF02', 'REF03', 'REF04']
POIDS_PIECES     = [0.40, 0.30, 0.20, 0.10]

# 6 mois × 30 jours × 24h × 2 sessions/h
N_SESSIONS       = 6 * 30 * 24 * 2
START_DATE       = '2024-01-01 06:00'

log.info("Configuration Gold Usine A chargée")
print(f"\n  Vitesse nominale   : {VITESSE_NOMINALE} tr/min  ← N = P×60/(2π×T)")
print(f"  Temps cycle nominal: {TEMPS_CYCLE_NOM} min      ← t = 60 000/N")
print(f"  Durée vie outil    : {DUREE_VIE_OUTIL} min      ← ISO 8688")

14:36:16 | INFO | Configuration Gold Usine A chargée



  Vitesse nominale   : 1500 tr/min  ← N = P×60/(2π×T)
  Temps cycle nominal: 40.0 min      ← t = 60 000/N
  Durée vie outil    : 250 min      ← ISO 8688


<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:10pt;font-weight:700;font-family:Georgia,serif;">
01 · Lecture Silver & structure temporelle
</span>
</div>

In [27]:
# ============================================================
# LECTURE SILVER
# On part du dataset propre Silver comme base de référence
# pour les distributions statistiques source.
# La structure temporelle Usine A est générée indépendamment.
# ============================================================

df_silver = pd.read_parquet(SILVER_FILE)
log.info("Silver chargé : %d lignes · %d colonnes", *df_silver.shape)

# Structure temporelle
timestamps = pd.date_range(start=START_DATE, periods=N_SESSIONS, freq='30min')
df = pd.DataFrame({'timestamp': timestamps})

df['usine_id']   = USINE_ID
df['machine_id'] = np.random.choice(MACHINES, size=N_SESSIONS)
df['type_piece'] = np.random.choice(TYPES_PIECES, size=N_SESSIONS, p=POIDS_PIECES)

def assigner_equipe(ts):
    h = ts.hour
    if 6 <= h < 14:    return 'Matin'
    elif 14 <= h < 22: return 'Apres-midi'
    else:               return 'Nuit'

df['equipe'] = df['timestamp'].apply(assigner_equipe)

print(f"\n  Sessions générées : {len(df):,}")
print(f"  Période           : {df['timestamp'].min()} → {df['timestamp'].max()}")
print(f"  Machines          : {df['machine_id'].value_counts().to_dict()}")
print(f"  Pièces            : {df['type_piece'].value_counts().to_dict()}")
print()
display(df.head(5))

14:36:16 | INFO | Silver chargé : 9985 lignes · 15 colonnes



  Sessions générées : 8,640
  Période           : 2024-01-01 06:00:00 → 2024-06-29 05:30:00
  Machines          : {'M01': 2193, 'M04': 2182, 'M02': 2134, 'M03': 2131}
  Pièces            : {'REF01': 3483, 'REF02': 2630, 'REF03': 1677, 'REF04': 850}



,timestamp,usine_id,machine_id,type_piece,equipe
0,2024-01-01 06:00:00,Usine_A,M03,REF02,Matin
1,2024-01-01 06:30:00,Usine_A,M04,REF04,Matin
2,2024-01-01 07:00:00,Usine_A,M01,REF02,Matin
3,2024-01-01 07:30:00,Usine_A,M03,REF02,Matin
4,2024-01-01 08:00:00,Usine_A,M03,REF03,Matin


<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:10pt;font-weight:700;font-family:Georgia,serif;">
02 · Génération des capteurs
</span>
</div>

In [28]:
# ============================================================
# GÉNÉRATION DES CAPTEURS — USINE A
# Distributions calées sur les statistiques Silver validées.
# L'usure outil est monotone croissante avec remise à zéro
# tardive à 90% : c'est le problème dominant Usine A.
# ============================================================

# Températures
df['temp_air_c']     = np.round(np.random.normal(TEMP_AIR_BASE, 1.5, N_SESSIONS), 2)
df['temp_process_c'] = np.round(df['temp_air_c'] + OFFSET_THERMIQUE
                                + np.random.normal(0, 0.8, N_SESSIONS), 2)
df['temp_process_delta'] = np.round(df['temp_process_c'] - df['temp_air_c'], 2)

# Capteurs machine
df['vitesse_rotation_rpm'] = np.clip(
    np.random.normal(VITESSE_NOMINALE, 60, N_SESSIONS).astype(int), 1200, 1800)
df['couple_nm'] = np.round(
    np.clip(np.random.normal(COUPLE_NOMINAL, 6, N_SESSIONS), 20, 65), 2)

# Usure outil — remplacement tardif à 90%
increments = {'REF01': (2.0,3.0),'REF02': (2.5,3.5),
              'REF03': (3.5,5.0),'REF04': (2.0,4.0)}

usure_actuelle = 0.0
usure_list, remplacement_list = [], []

for i in range(N_SESSIONS):
    lo, hi = increments[df.loc[i,'type_piece']]
    usure_actuelle += np.random.uniform(lo, hi)
    if usure_actuelle >= DUREE_VIE_OUTIL * 0.90:
        remplacement_list.append(1)
        usure_actuelle = 0.0
    else:
        remplacement_list.append(0)
    usure_list.append(round(usure_actuelle, 1))

df['usure_outil_min']    = usure_list
df['remplacement_outil'] = remplacement_list
df['usure_cumulee_pct']  = np.round(
    (df['usure_outil_min'] / DUREE_VIE_OUTIL * 100).clip(0,100), 2)

print(f"  Temp air          : {df['temp_air_c'].mean():.1f}°C ± {df['temp_air_c'].std():.1f}")
print(f"  Vitesse moyenne   : {df['vitesse_rotation_rpm'].mean():.0f} tr/min")
print(f"  Remplacements     : {sum(remplacement_list)} outils sur 6 mois")
print(f"  Usure max         : {df['usure_outil_min'].max():.1f} min")

  Temp air          : 22.0°C ± 1.5
  Vitesse moyenne   : 1498 tr/min
  Remplacements     : 115 outils sur 6 mois
  Usure max         : 225.0 min


<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:10pt;font-weight:700;font-family:Georgia,serif;">
03 · Pannes & maintenance
</span>
</div>

In [29]:
# ============================================================
# PANNES — USINE A
# P(panne_usure) = (usure_pct/100)² × 0.15
# Croissance quadratique validée sur le diagnostic Bronze.
# ============================================================

prob_usure    = (df['usure_cumulee_pct'] / 100) ** 2 * 0.15
panne_usure   = ((np.random.random(N_SESSIONS) < prob_usure) &
                 (df['usure_cumulee_pct'] > 70)).astype(int)
panne_surge   = (np.random.random(N_SESSIONS) < 0.005).astype(int)
panne_alea    = (np.random.random(N_SESSIONS) < 0.001).astype(int)
panne_machine = ((panne_usure==1)|(panne_surge==1)|(panne_alea==1)).astype(int)

df['panne_machine']     = panne_machine
df['panne_usure_outil'] = panne_usure
df['panne_thermique']   = 0
df['panne_puissance']   = 0
df['panne_surcharge']   = panne_surge
df['panne_aleatoire']   = panne_alea

def type_panne(row):
    if row['panne_usure_outil']: return 'Usure outil'
    if row['panne_surcharge']:   return 'Surcharge'
    if row['panne_aleatoire']:   return 'Aleatoire'
    return 'Aucune'

df['type_panne'] = df.apply(type_panne, axis=1)

durees = []
for _, row in df.iterrows():
    if row['panne_usure_outil']: durees.append(round(np.random.uniform(60,180),1))
    elif row['panne_surcharge']: durees.append(round(np.random.uniform(30,90),1))
    elif row['panne_aleatoire']: durees.append(round(np.random.uniform(15,45),1))
    else: durees.append(0.0)

df['duree_arret_min']  = durees
df['nb_pannes_7j']     = df['panne_machine'].rolling(336, min_periods=1).sum().astype(int)
df['intervention_maintenance'] = ((df['panne_machine']==1)|(df['nb_pannes_7j']>5)).astype(int)

print(f"  Taux panne global   : {df['panne_machine'].mean()*100:.2f}%")
print(f"  Pannes usure outil  : {df['panne_usure_outil'].sum()}")
print(f"  Durée arrêt moy     : {df[df['duree_arret_min']>0]['duree_arret_min'].mean():.1f} min")
print(f"  Interventions maint.: {df['intervention_maintenance'].sum()}")

  Taux panne global   : 2.80%
  Pannes usure outil  : 184
  Durée arrêt moy     : 105.8 min
  Interventions maint.: 7688


<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:10pt;font-weight:700;font-family:Georgia,serif;">
04 · KPIs production
</span>
</div>

In [30]:
# ============================================================
# KPIs PRODUCTION — USINE A
#
# Temps cycle : t = t_nominal × (1 + 0.003 × usure_pct) + bruit
# Rebut       : τ = τ_base + α×usure²/D + β×panne + bruit
# Énergie     : E = (N×T/9550) × 0.5 + bruit
# ============================================================

df['temps_cycle_min'] = np.round(
    (TEMPS_CYCLE_NOM * (1 + 0.003 * df['usure_cumulee_pct'])
     + np.random.normal(0, 1.5, N_SESSIONS)).clip(lower=20), 2)

df['derive_cycle_pct'] = np.round(
    (df['temps_cycle_min'] - TEMPS_CYCLE_NOM) / TEMPS_CYCLE_NOM * 100, 2)

df['pieces_produites'] = (30 / df['temps_cycle_min']).astype(int).clip(lower=1)
df.loc[df['panne_machine']==1, 'pieces_produites'] = 0

df['taux_rebut'] = np.round(
    (REBUT_BASE
     + 0.0003 * (df['usure_outil_min']**2) / DUREE_VIE_OUTIL
     + 0.15 * df['panne_machine']
     + np.random.normal(0, 0.004, N_SESSIONS)).clip(0,1), 4)

df['pieces_rebutees'] = (df['pieces_produites'] * df['taux_rebut']).astype(int)

df['indice_stabilite_process'] = np.round(
    (0.92 - 0.30*df['panne_machine']
     - 0.001*df['usure_cumulee_pct']
     + np.random.normal(0, 0.03, N_SESSIONS)).clip(0,1), 4)

df['consommation_kwh'] = np.round(
    ((df['vitesse_rotation_rpm'] * df['couple_nm']) / 9550 * 0.5
     + np.random.normal(0, 0.4, N_SESSIONS)).clip(lower=0), 2)

mask_appro = (df['timestamp'] >= '2024-05-10') & (df['timestamp'] <= '2024-05-15')
df['rupture_appro'] = mask_appro.astype(int)
df.loc[mask_appro, 'pieces_produites'] = 0

print(f"  Temps cycle moyen   : {df['temps_cycle_min'].mean():.2f} min (nominal : {TEMPS_CYCLE_NOM})")
print(f"  Dérive cycle max    : {df['derive_cycle_pct'].max():.1f}%")
print(f"  Taux rebut moyen    : {df['taux_rebut'].mean()*100:.2f}%")
print(f"  Pièces produites    : {df['pieces_produites'].sum():,}")
print(f"  Conso énergie moy   : {df['consommation_kwh'].mean():.2f} kWh")

  Temps cycle moyen   : 45.36 min (nominal : 40.0)
  Dérive cycle max    : 36.6%
  Taux rebut moyen    : 4.44%
  Pièces produites    : 8,168
  Conso énergie moy   : 3.14 kWh


<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:10pt;font-weight:700;font-family:Georgia,serif;">
05 · TRS & Limites de contrôle SPC
</span>
</div>

In [31]:
# ============================================================
# TRS = Disponibilité × Performance × Qualité
# UCL/LCL : méthode SPC 3σ depuis baseline janvier 2024
# ============================================================

df['disponibilite'] = (df['panne_machine'] == 0).astype(float)
df['performance']   = np.round((df['vitesse_rotation_rpm']/VITESSE_NOMINALE).clip(upper=1.0), 4)
df['qualite']       = np.round(1 - df['taux_rebut'], 4)
df['TRS']           = np.round(df['disponibilite']*df['performance']*df['qualite'], 4)

# Baseline : janvier 2024, sessions sans panne
baseline = df[(df['timestamp'] < '2024-02-01') & (df['panne_machine']==0)]

def calcul_limites(serie, nom, n_sigma=3):
    mu, sigma = serie.mean(), serie.std()
    return {
        'kpi'    : nom,
        'moyenne': round(mu, 4),
        'sigma'  : round(sigma, 4),
        'ucl'    : round(mu + n_sigma*sigma, 4),
        'lcl'    : round(max(mu - n_sigma*sigma, 0), 4),
        'uwl'    : round(mu + 2*sigma, 4),
        'lwl'    : round(max(mu - 2*sigma, 0), 4),
    }

kpis = {
    'TRS'                : baseline['TRS'],
    'taux_rebut'         : baseline['taux_rebut'],
    'temps_cycle_min'    : baseline['temps_cycle_min'],
    'temp_process_delta' : baseline['temp_process_delta'],
    'consommation_kwh'   : baseline['consommation_kwh'],
    'indice_stabilite'   : baseline['indice_stabilite_process'],
}

limites = {nom: calcul_limites(serie, nom) for nom, serie in kpis.items()}
df_lim  = pd.DataFrame(limites.values()).set_index('kpi')

print(f"  TRS moyen global : {df['TRS'].mean()*100:.1f}%")
print(f"  TRS moyen S1     : {df[df['timestamp']<'2024-02-01']['TRS'].mean()*100:.1f}%")
print(f"  TRS moyen S6     : {df[df['timestamp']>='2024-06-01']['TRS'].mean()*100:.1f}%")
print()
display(df_lim)

  TRS moyen global : 91.8%
  TRS moyen S1     : 92.2%
  TRS moyen S6     : 91.8%



,moyenne,sigma,ucl,lcl,uwl,lwl
kpi,,,,,,
TRS,0.94,0.03,1.03,0.86,1.00,0.89
taux_rebut,0.04,0.02,0.09,0.00,0.08,0.00
temps_cycle_min,45.28,3.37,55.39,35.17,52.02,38.54
temp_process_delta,10.00,0.80,12.40,7.59,11.60,8.39
consommation_kwh,3.12,0.63,5.01,1.24,4.38,1.87
indice_stabilite,0.88,0.04,1.00,0.76,0.96,0.80


<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:10pt;font-weight:700;font-family:Georgia,serif;">
06 · Visualisations Gold — dérives Usine A
</span>
</div>

In [32]:
# ── Plot 1 : TRS sur 6 mois avec moyenne mobile ──────────────
sample = df.iloc[::3].copy()
df['TRS_7j'] = df['TRS'].rolling(336, min_periods=1).mean()

fig1 = go.Figure()
fig1.add_trace(go.Scatter(
    x=sample['timestamp'], y=sample['TRS'],
    name='TRS session', line=dict(color='#c7d2fe', width=0.8), opacity=0.6,
    hovertemplate='%{x|%d %b %H:%M}<br>TRS : %{y:.3f}<extra></extra>'
))
fig1.add_trace(go.Scatter(
    x=df['timestamp'], y=df['TRS_7j'],
    name='Moyenne mobile 7j', line=dict(color='#6366f1', width=2.5),
    hovertemplate='%{x|%d %b}<br>TRS moy 7j : %{y:.3f}<extra></extra>'
))
fig1.add_hline(y=limites['TRS']['lcl'], line_dash='dot', line_color='#ef4444',
               line_width=1.2,
               annotation_text=f"LCL = {limites['TRS']['lcl']:.3f}",
               annotation_font_size=9, annotation_font_color='#ef4444',
               annotation_position='bottom right')
fig1.update_layout(
    **LAYOUT,
    title=dict(text='Usine A - TRS sur 6 mois (dégradation progressive)',
               font=dict(size=12, color='#111827'), x=0.01),
    xaxis=dict(gridcolor='#f3f4f6', linecolor='#e5e7eb',
               tickfont=dict(size=10, color='#9ca3af'), tickformat='%d %b'),
    yaxis=dict(gridcolor='#f3f4f6', linecolor='#e5e7eb',
               tickfont=dict(size=10, color='#9ca3af'), title='TRS', range=[0,1.05]),
    legend=dict(bgcolor='rgba(0,0,0,0)', borderwidth=0,
                font=dict(size=10), orientation='h', y=1.08),
    height=380, margin=dict(l=50,r=20,t=60,b=40),
    hovermode='x unified'
)
fig1.show()

In [35]:
# ── Plot 2 : Taux de rebut vs usure outil ────────────────────
fig2 = make_subplots(rows=2, cols=1, shared_xaxes=True,
                     row_heights=[0.6,0.4], vertical_spacing=0.06)

fig2.add_trace(go.Scatter(
    x=sample['timestamp'], y=sample['taux_rebut'],
    name='Taux de rebut', line=dict(color='#ef4444', width=1.2),
    fill='tozeroy', fillcolor='rgba(239,68,68,0.06)',
    hovertemplate='%{x|%d %b %H:%M}<br>Rebut : %{y:.3f}<extra></extra>'
), row=1, col=1)

fig2.add_hline(y=limites['taux_rebut']['ucl'], row=1, col=1,
               line_dash='dot', line_color='#ef4444', line_width=1.2,
               annotation_text=f"UCL = {limites['taux_rebut']['ucl']:.3f}",
               annotation_font_size=9, annotation_font_color='#ef4444',
               annotation_position='top right')

fig2.add_trace(go.Scatter(
    x=sample['timestamp'], y=sample['usure_cumulee_pct'],
    name='Usure outil (%)', line=dict(color='#f97316', width=1.2),
    fill='tozeroy', fillcolor='rgba(249,115,22,0.06)',
    hovertemplate='%{x|%d %b %H:%M}<br>Usure : %{y:.1f}%<extra></extra>'
), row=2, col=1)

fig2.add_hline(y=90, row=2, col=1, line_dash='dot', line_color='#ef4444',
               line_width=1, annotation_text='Seuil remplacement 90%',
               annotation_font_size=9, annotation_font_color='#ef4444',
               annotation_position='top right')

fig2.update_layout(
    **LAYOUT,
    title=dict(text='Usine A - Taux de rebut vs Usure outil cumulée',
               font=dict(size=12, color='#111827'), x=0.01),
    height=480, margin=dict(l=50,r=20,t=60,b=40),
    legend=dict(bgcolor='rgba(0,0,0,0)', borderwidth=0,
                font=dict(size=10), orientation='h', y=1.08),
    hovermode='x unified'
)
fig2.update_xaxes(gridcolor='#f3f4f6', linecolor='#e5e7eb',
                  tickfont=dict(size=9, color='#9ca3af'), tickformat='%d %b')
fig2.update_yaxes(gridcolor='#f3f4f6', linecolor='#e5e7eb',
                  tickfont=dict(size=9, color='#9ca3af'))
fig2.show()

<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:10pt;font-weight:700;font-family:Georgia,serif;">
07 · Export Gold
</span>
</div>

In [34]:
# ============================================================
# EXPORT GOLD — Parquet + JSON limites
# ============================================================

def json_serial(obj):
    if isinstance(obj, np.integer): return int(obj)
    if isinstance(obj, np.floating): return float(obj)
    if isinstance(obj, np.ndarray):  return obj.tolist()
    raise TypeError(f"Type {type(obj)} non sérialisable")

gold_file   = GOLD_PATH / 'usine_A.parquet'
lim_file    = GOLD_PATH / 'limites_usine_A.json'

df.to_parquet(gold_file, index=False)

with open(lim_file, 'w') as f:
    json.dump(limites, f, indent=2, default=json_serial)

log.info("Gold Usine A exporté : %s", gold_file)

print(f"\n{'='*55}")
print(f"  GOLD OUTPUTS — USINE A")
print(f"{'='*55}")
print(f"  ✓ {gold_file}")
print(f"  ✓ {lim_file}")
print(f"  Shape : {df.shape[0]:,} lignes · {df.shape[1]} colonnes")
print(f"  TRS moyen : {df['TRS'].mean()*100:.1f}%")
print(f"  Taux rebut moyen : {df['taux_rebut'].mean()*100:.2f}%")
print(f"{'='*55}")

14:36:17 | INFO | Gold Usine A exporté : C:\Users\j.innouri\OneDrive - pellencst.com\Bureau\MSPR_TPRE831\data\gold\usine_A.parquet



  GOLD OUTPUTS — USINE A
  ✓ C:\Users\j.innouri\OneDrive - pellencst.com\Bureau\MSPR_TPRE831\data\gold\usine_A.parquet
  ✓ C:\Users\j.innouri\OneDrive - pellencst.com\Bureau\MSPR_TPRE831\data\gold\limites_usine_A.json
  Shape : 8,640 lignes · 36 colonnes
  TRS moyen : 91.8%
  Taux rebut moyen : 4.44%
